# LeetCode 837 — New 21 Game bằng Python

Link đề: https://leetcode.com/problems/new-21-game/ · Độ khó: **Medium** · Chủ đề: Math, Dynamic Programming, Sliding Window, Probability and Statistics

Skeleton LeetCode:

```python
class Solution:
    def new21Game(self, n: int, k: int, maxPts: int) -> float:
        ...
```

Ghi chú: kết quả là **số thực**, đề chấp nhận sai số `10^-5`. Notebook dùng `fractions.Fraction` để tính **xác suất chính xác** làm chuẩn (trên input vừa phải), so sánh các cài đặt `float` bằng **dung sai** (`abs(a − b) <= 1e-9`), và kiểm tra thống kê bằng mô phỏng Monte Carlo với RNG tự dựng có seed.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: Alice bắt đầu `0` điểm, **còn `< k` điểm thì còn bốc**; mỗi lần bốc được số nguyên đều trong `[1, maxPts]`; dừng khi đạt `≥ k`. Tính xác suất điểm cuối cùng `≤ n`.
2. Nắm các công cụ Python: quy hoạch động theo điểm, **tổng trượt (sliding window sum)**, `fractions.Fraction` (xác suất chính xác), so sánh số thực với dung sai, mô phỏng ngẫu nhiên có seed.
3. Đi từ **đệ quy thuần (mũ)** → **DP theo điểm với tổng tính lại** (`O(k · maxPts)`) → **DP với tổng trượt** (`O(k + maxPts)`) → **DP xuôi: xác suất "từng đạt điểm `i`"** rồi cộng các điểm kết thúc `≤ n`.
3b. Hiểu hai cách nhìn: **ngược** `f(x)` = xác suất thành công từ điểm `x`, và **xuôi** `p(i)` = xác suất đi qua điểm `i`.
4. Hiểu mấu chốt: `f(x) = (f(x+1) + … + f(x+maxPts)) / maxPts` cho `x < k`, và hai biểu thức liên tiếp `f(x)`, `f(x−1)` chia sẻ gần hết số hạng nên tổng cập nhật **O(1)**: thêm `f(x)`, bớt `f(x + maxPts)`.
5. Tránh các lỗi kinh điển: sai biên (`x = k` đã dừng), quên chia `maxPts`, trượt cửa sổ lệch một, đệ quy quá sâu / chậm `O(k·maxPts)`, và quên các ca đặc biệt `k = 0`, `n ≥ k + maxPts − 1`.

## 2. Đề bài & ví dụ

Alice chơi một trò gần giống bài bài "21". Alice bắt đầu với `0` điểm và **bốc số** khi còn **ít hơn `k` điểm**. Mỗi lần bốc, cô nhận một số nguyên điểm **ngẫu nhiên** trong khoảng `[1, maxPts]` (các giá trị đồng khả năng, các lần bốc độc lập).

Alice **dừng bốc** khi có `k` điểm **trở lên**.

Trả về **xác suất** Alice có `n` điểm **trở xuống** khi dừng. Đáp án sai khác `10^-5` được chấp nhận.

**Ví dụ 1:** `n = 10, k = 1, maxPts = 10` → `1.00000`. Alice bốc đúng một lần (điểm từ 1 đến 10) rồi dừng (vì `≥ 1`); mọi kết quả `≤ 10`.

**Ví dụ 2:** `n = 6, k = 1, maxPts = 10` → `0.60000`. Bốc một lần; `6/10` khả năng `≤ 6`.

**Ví dụ 3:** `n = 21, k = 17, maxPts = 10` → `0.73278`.

**Ràng buộc:** `0 <= k <= n <= 10^4`; `1 <= maxPts <= 10^4`.

Chú ý: điểm cuối cùng nằm trong `[k, k + maxPts − 1]` (Alice dừng ngay khi `≥ k`, và lần bốc cuối từ điểm `≤ k − 1` thêm tối đa `maxPts`).

In [1]:
import sys
import math
import time
import random
from fractions import Fraction
from functools import cache
from typing import List

for n, k, m in [(10, 1, 10), (6, 1, 10), (21, 17, 10)]:
    print(f"n={n}, k={k}, maxPts={m}: điểm cuối nằm trong [{k}, {k + m - 1}]", "| chắc chắn <= n:", n >= k + m - 1)

n=10, k=1, maxPts=10: điểm cuối nằm trong [1, 10] | chắc chắn <= n: True
n=6, k=1, maxPts=10: điểm cuối nằm trong [1, 10] | chắc chắn <= n: False
n=21, k=17, maxPts=10: điểm cuối nằm trong [17, 26] | chắc chắn <= n: False


## 3. Kiến thức nền

### 3.1 Xác suất và kỳ vọng "ngược": `f(x)`

Gọi `f(x)` là xác suất **cuối cùng có `≤ n` điểm**, biết hiện đang có `x` điểm. Khi `x ≥ k` Alice đã dừng, nên `f(x) = 1` nếu `x ≤ n`, và `0` nếu `x > n`. Khi `x < k` cô bốc thêm, nhận `i ∈ {1, …, maxPts}` với xác suất `1/maxPts` mỗi giá trị: `f(x) = (1/maxPts) · Σ_{i=1}^{maxPts} f(x + i)` (luật xác suất toàn phần). Đáp án là `f(0)`.

### 3.2 Xác suất "xuôi": `p(i)`

Gọi `p(i)` là xác suất Alice **có lúc có đúng `i` điểm** (đi qua `i`). `p(0) = 1`; với `i ≥ 1`, để đến `i` cô phải ở điểm `j < i` (và `j < k`, vì còn bốc) rồi bốc đúng `i − j ≤ maxPts`: `p(i) = (1/maxPts) · Σ_{j = max(0, i − maxPts)}^{min(i − 1, k − 1)} p(j)`. Alice dừng tại các điểm `i ≥ k` (và mỗi lần đến `i ≥ k` là kết thúc), nên xác suất kết thúc đúng tại `i` là `p(i)`; đáp án `= Σ_{i=k}^{n} p(i)`.

### 3.3 Tổng trượt (sliding window sum)

Hai tổng liên tiếp `S(x) = Σ_{i=1}^{m} f(x + i)` và `S(x − 1) = Σ_{i=1}^{m} f(x − 1 + i) = f(x) + f(x+1) + … + f(x + m − 1)` chỉ khác nhau: `S(x − 1) = S(x) + f(x) − f(x + m)`. Nghĩa là cập nhật tổng trong `O(1)` bằng cách **thêm số mới vào, bớt số cũ ra**, thay vì cộng lại `m` số. Đó là kỹ thuật "cửa sổ trượt".

In [2]:
vals = [3, 1, 4, 1, 5, 9, 2, 6]
m = 3
window = sum(vals[:m])
sums = [window]
for i in range(m, len(vals)):
    window += vals[i] - vals[i - m]            # thêm phần tử mới, bớt phần tử rời cửa sổ
    sums.append(window)
print(sums, "| cộng lại từ đầu:", [sum(vals[i:i + m]) for i in range(len(vals) - m + 1)])

[8, 6, 10, 15, 16, 17] | cộng lại từ đầu: [8, 6, 10, 15, 16, 17]


### 3.4 `Fraction` và dung sai cho số thực

`Fraction` cho xác suất **chính xác**; `float` có sai số làm tròn `~10^-16` mỗi phép tính. Test số thực: dùng `abs(a − b) <= eps`, không dùng `==`. Với tổng trượt, sai số cộng dồn qua khoảng `k` bước, cỡ `10^-12` — rất nhỏ so với dung sai đề `10^-5`.

In [3]:
print(0.1 + 0.2 == 0.3, abs(0.1 + 0.2 - 0.3) <= 1e-12)
x = sum(Fraction(1, 10) for _ in range(10))
y = sum(0.1 for _ in range(10))
print(x == 1, y == 1.0, abs(y - 1.0))

False True
True True 0.0


### 3.5 Đệ quy có nhớ vs bảng DP

`@cache` nhớ kết quả đệ quy; bảng DP điền từ trạng thái "gần kết thúc" về `x = 0`. Ở bài này `f(x)` phụ thuộc `f(x+1..x+m)` — các **chỉ số lớn hơn** — nên điền **giảm dần** từ `x = k − 1` về `0`. Đệ quy sâu tới `k` tầng (mỗi lần bốc tăng ít nhất 1 điểm), có thể tới `10^4`: vượt giới hạn đệ quy mặc định `1000` của Python.

In [4]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit())

@cache
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80), "| bảng DP cùng ý tưởng nhưng không dùng đệ quy")

giới hạn đệ quy mặc định: 1000
23416728348467685 | bảng DP cùng ý tưởng nhưng không dùng đệ quy


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `n = 6`, `k = 1`, `maxPts = 10`. Alice có `0 < 1` điểm nên bốc **một** lần, nhận `i ∈ {1..10}`; có `i ≥ 1` nên dừng. Điểm cuối `= i`; `≤ 6` khi `i ∈ {1..6}`: `6/10 = 0,6`.

Ví dụ nhỏ tự dựng (cũng dùng làm bài tập): `n = 2`, `k = 2`, `maxPts = 2`. Điểm bốc `{1, 2}`:

- Điểm đang `0`: bốc `2` (xác suất `½`) → `2 ≥ k` dừng, `2 ≤ n` ✓. Bốc `1` (`½`) → ở `1 < 2` bốc tiếp: nhận `1` → `2` dừng ✓ (`½`), nhận `2` → `3` dừng, `3 > n` ✗ (`½`).
- `P = ½ · 1 + ½ · (½ · 1 + ½ · 0) = ½ + ¼ = 0,75`.

Theo công thức: `f(2) = 1`, `f(3) = 0` (đã dừng, `3 > n`), `f(1) = (f(2) + f(3))/2 = 0,5`, `f(0) = (f(1) + f(2))/2 = 0,75` ✓.

**Hai ca đặc biệt cho đáp án ngay:**

- `k = 0`: Alice không bốc lần nào, có `0 ≤ n` điểm → `1`.
- `n ≥ k + maxPts − 1`: điểm cuối tối đa `k − 1 + maxPts ≤ n` nên chắc chắn `≤ n` → `1`.

In [5]:
def f_values(n, k, m):
    """Bảng f(x) chính xác (Fraction) cho x = 0 .. k+m-1, đúng định nghĩa."""
    f = {}
    for x in range(k + m - 1, -1, -1):
        if x >= k:
            f[x] = Fraction(1) if x <= n else Fraction(0)
        else:
            f[x] = sum(f[x + i] for i in range(1, m + 1)) / m
    return f


f = f_values(2, 2, 2)
print({x: str(v) for x, v in f.items()})
print("f(0) =", f[0], "=", float(f[0]))
assert f[0] == Fraction(3, 4)
f = f_values(6, 1, 10)
print("n=6, k=1, m=10: f(0) =", f[0])
assert f[0] == Fraction(3, 5)

{3: '0', 2: '1', 1: '1/2', 0: '3/4'}
f(0) = 3/4 = 0.75
n=6, k=1, m=10: f(0) = 3/5


## 5. Cách 1 — Brute Force: đệ quy thuần theo định nghĩa

**Ý tưởng:** viết đúng định nghĩa `f(x)` bằng đệ quy **không nhớ**, dùng `Fraction` để chính xác: khi `x ≥ k` trả `1` hoặc `0`; ngược lại trung bình của `maxPts` lời gọi `f(x + i)`.

In [6]:
def game_brute(n, k, m):
    calls = [0]

    def f(x):
        calls[0] += 1
        if x >= k:
            return Fraction(1) if x <= n else Fraction(0)
        return sum(f(x + i) for i in range(1, m + 1)) / m

    res = f(0)
    game_brute.calls = calls[0]
    return res


for n, k, m in [(2, 2, 2), (6, 1, 10), (10, 1, 10), (7, 5, 3), (9, 6, 4), (12, 8, 3)]:
    r = game_brute(n, k, m)
    print(f"n={n:2d} k={k} m={m}: {r!s:>10} = {float(r):.6f}  ({game_brute.calls:>6,d} lời gọi)")
assert game_brute(6, 1, 10) == Fraction(3, 5) and game_brute(10, 1, 10) == 1

n= 2 k=2 m=2:        3/4 = 0.750000  (     5 lời gọi)
n= 6 k=1 m=10:        3/5 = 0.600000  (    11 lời gọi)
n=10 k=1 m=10:          1 = 1.000000  (    11 lời gọi)
n= 7 k=5 m=3:          1 = 1.000000  (    46 lời gọi)
n= 9 k=6 m=4:          1 = 1.000000  (   125 lời gọi)
n=12 k=8 m=3:          1 = 1.000000  (   289 lời gọi)


**Độ phức tạp:** số lời gọi tăng **mũ** theo `k` (mỗi nút có `maxPts` nhánh, độ sâu tới `k`), tức `O(maxPts^k)` trong trường hợp xấu; bộ nhớ `O(k)` (độ sâu). Chỉ dùng cho input rất nhỏ làm oracle.

**Nút thắt:** `f(x)` bị tính lại nhiều lần (cùng `x` đến từ nhiều chuỗi bốc khác nhau). Chỉ có `k + maxPts` giá trị `x` khác nhau.

## 6. Ý tưởng thuật toán / bước đột phá

1. **Nhớ kết quả (DP):** mỗi `f(x)` tính **một lần**, điền từ `x = k + maxPts − 1` ngược về `0`. Mỗi `f(x)` cộng `maxPts` số: `O(k · maxPts)` — với `k = maxPts = 10^4` là `10^8`, quá chậm trong Python.
2. **Tổng trượt:** `S(x−1) = S(x) + f(x) − f(x + m)` (mục 3.3) cho `f` mỗi `x` trong `O(1)`: tổng `O(k + maxPts)`.
3. **Ca đặc biệt:** `k = 0` hoặc `n ≥ k + maxPts − 1` trả `1`.
4. **Cách xuôi:** `p(i)` với tổng trượt `Σ p(j)` — cùng độ phức tạp, tính "xác suất đi qua điểm `i`" rồi cộng các điểm kết thúc `≤ n`.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** DP đủ tổng (`O(k · maxPts)`), **Cách 3** DP tổng trượt (`O(k + maxPts)`, cách chuẩn), **Cách 4** DP xuôi `p(i)`.

### Cách 2 — DP theo điểm, mỗi ô cộng đủ `maxPts` số

**Giải thích từng đoạn:**

1. `f[x]` cho `x = k .. k + m − 1`: `1` nếu `x ≤ n`, ngược lại `0`.
2. Với `x = k − 1` giảm dần về `0`: `f[x] = (f[x+1] + … + f[x+m]) / m` (cộng đủ `m` số).
3. Đáp án `f[0]`.

In [7]:
def game_dp(n, k, m):
    f = [0.0] * (k + m)
    for x in range(k, min(n, k + m - 1) + 1):
        f[x] = 1.0
    for x in range(k - 1, -1, -1):
        f[x] = sum(f[x + 1:x + m + 1]) / m
    return f[0]


print(game_dp(2, 2, 2), game_dp(6, 1, 10), game_dp(21, 17, 10))
assert abs(game_dp(2, 2, 2) - 0.75) < 1e-12 and abs(game_dp(6, 1, 10) - 0.6) < 1e-12 and abs(game_dp(21, 17, 10) - 0.73278) < 1e-5

0.75 0.6 0.7327777870686081


In [8]:
# Code nộp LeetCode cho Cách 2 (chậm O(k*maxPts), chỉ để học)
class SolutionDP:
    def new21Game(self, n: int, k: int, maxPts: int) -> float:
        f = [0.0] * (k + maxPts)
        for x in range(k, min(n, k + maxPts - 1) + 1):
            f[x] = 1.0
        for x in range(k - 1, -1, -1):
            f[x] = sum(f[x + 1:x + maxPts + 1]) / maxPts
        return f[0]

**Độ phức tạp:** `O(k · maxPts)` thời gian (cộng `maxPts` số cho mỗi `x < k`), `O(k + maxPts)` bộ nhớ. Với `k = maxPts = 10^4`: `10^8` phép cộng — bản viết bằng vòng lặp Python thuần sẽ quá thời gian; ngay cả khi dùng `sum(f[a:b])` (cộng bằng C) vẫn chậm hơn hẳn tổng trượt (hàng giây so với vài mili-giây).

### Cách 3 — DP với tổng trượt (cách chuẩn)

**Giải thích từng đoạn:**

1. Ca đặc biệt: `k == 0` hoặc `n >= k + maxPts − 1` → `1.0`.
2. `f[x] = 1` cho `x ∈ [k, n]` (và `0` cho `x > n`).
3. `s` = tổng cửa sổ `f[k] + … + f[k + maxPts − 1]` (đây là `S(k − 1)`).
4. Với `x = k − 1` về `0`: `f[x] = s / maxPts`; rồi cập nhật cửa sổ cho `x − 1`: `s += f[x] − f[x + maxPts]`.
5. Trả `f[0]`.

In [9]:
def game_window(n, k, m):
    if k == 0 or n >= k + m - 1:
        return 1.0
    f = [0.0] * (k + m)
    for x in range(k, n + 1):
        f[x] = 1.0
    s = float(sum(f[k:k + m]))                 # S(k-1) = f[k] + ... + f[k+m-1]
    for x in range(k - 1, -1, -1):
        f[x] = s / m
        s += f[x] - f[x + m]                   # cửa sổ trượt: thêm f[x], bớt f[x+m]
    return f[0]


print(game_window(2, 2, 2), game_window(6, 1, 10), game_window(21, 17, 10), game_window(10, 1, 10))
assert abs(game_window(2, 2, 2) - 0.75) < 1e-12 and abs(game_window(6, 1, 10) - 0.6) < 1e-12
assert abs(game_window(21, 17, 10) - 0.73278) < 1e-5 and game_window(10, 1, 10) == 1.0

0.75 0.6 0.7327777870686075 1.0


In [10]:
# Code nộp LeetCode cho Cách 3 (đáp án chính)
class SolutionWindow:
    def new21Game(self, n: int, k: int, maxPts: int) -> float:
        if k == 0 or n >= k + maxPts - 1:
            return 1.0
        f = [0.0] * (k + maxPts)
        for x in range(k, n + 1):
            f[x] = 1.0
        s = float(sum(f[k:k + maxPts]))
        for x in range(k - 1, -1, -1):
            f[x] = s / maxPts
            s += f[x] - f[x + maxPts]
        return f[0]

**Độ phức tạp:** `O(k + maxPts)` thời gian (mỗi `x` làm `O(1)` phép), `O(k + maxPts)` bộ nhớ. Với `10^4`: tức thì.

### Cách 4 — DP xuôi: xác suất "đi qua" mỗi điểm

**Ý tưởng khác:** thay vì tính xác suất thành công từ mỗi điểm (ngược), tính **xác suất đi qua điểm `i`**, `p(i)`, từ `i = 0` tăng dần: `p(0) = 1`, `p(i) = (1/m) · Σ_{j = max(0, i−m)}^{min(i−1, k−1)} p(j)`. Alice dừng ở điểm `i ≥ k`, nên **xác suất kết thúc tại `i` bằng `p(i)`** (với `k ≤ i ≤ k + m − 1`), và đáp án là `Σ_{i = k}^{min(n, k+m−1)} p(i)`. Cửa sổ `Σ p(j)` cũng cập nhật `O(1)`: thêm `p(i)` nếu `i < k` (điểm còn bốc được), bớt `p(i − m)` nếu cần.

In [11]:
def game_forward(n, k, m):
    if k == 0:
        return 1.0
    top = k + m - 1                                  # điểm cuối tối đa
    p = [0.0] * (top + 1)
    p[0] = 1.0
    window = 1.0                                     # tổng p(j) cho các j trong [i-m, i-1] mà j < k
    ans = 0.0
    for i in range(1, top + 1):
        p[i] = window / m
        if i < k:
            window += p[i]                           # i còn bốc tiếp: tham gia cửa sổ
        elif i <= n:
            ans += p[i]                              # kết thúc tại i <= n
        if i - m >= 0 and i - m < k:
            window -= p[i - m]                       # p(i-m) rời cửa sổ
    return ans


print(game_forward(2, 2, 2), game_forward(6, 1, 10), game_forward(21, 17, 10), game_forward(10, 1, 10), game_forward(0, 0, 5))
assert abs(game_forward(2, 2, 2) - 0.75) < 1e-12 and abs(game_forward(6, 1, 10) - 0.6) < 1e-12
assert abs(game_forward(21, 17, 10) - 0.73278) < 1e-5 and abs(game_forward(10, 1, 10) - 1.0) < 1e-12 and game_forward(0, 0, 5) == 1.0

0.75 0.6 0.7327777870686082 0.9999999999999999 1.0


In [12]:
# Code nộp LeetCode cho Cách 4
class SolutionForward:
    def new21Game(self, n: int, k: int, maxPts: int) -> float:
        if k == 0:
            return 1.0
        top = k + maxPts - 1
        p = [0.0] * (top + 1)
        p[0] = 1.0
        window, ans = 1.0, 0.0
        for i in range(1, top + 1):
            p[i] = window / maxPts
            if i < k:
                window += p[i]
            elif i <= n:
                ans += p[i]
            if 0 <= i - maxPts < k:
                window -= p[i - maxPts]
        return ans

**Độ phức tạp:** `O(k + maxPts)` thời gian và bộ nhớ. Cách này cho thêm một đầu ra: **phân phối điểm cuối** `p(k), …, p(k + maxPts − 1)` (bài tập 5).

## 8. TRACE từng bước

**Trace Cách 3** với `n = 21, k = 17, maxPts = 10`. Ca đặc biệt không áp dụng (`21 < 17 + 10 − 1 = 26`). `f[x] = 1` cho `x = 17..21`, `f[x] = 0` cho `x = 22..26`. Cửa sổ ban đầu `s = f[17] + … + f[26] = 5`. Mỗi bước `x = 16, 15, …, 0`: `f[x] = s / 10`, `s += f[x] − f[x + 10]`:

| `x` | `s` (cửa sổ trước) | `f[x] = s/10` | bớt `f[x+10]` | `s` sau |
|---|---|---|---|---|
| 16 | 5 | 0,5 | `f[26] = 0` | 5,5 |
| 15 | 5,5 | 0,55 | `f[25] = 0` | 6,05 |
| 14 | 6,05 | 0,605 | `f[24] = 0` | 6,655 |
| 13 | 6,655 | 0,6655 | `f[23] = 0` | 7,3205 |
| 12 | 7,3205 | 0,73205 | `f[22] = 0` | 8,05255 |
| 11 | 8,05255 | 0,805255 | `f[21] = 1` | 7,857805 |
| … | … | … | … | … |

Hàm trace in toàn bộ các bước (tới `x = 0`, `f[0] ≈ 0,73278`):

In [13]:
def trace_window(n, k, m):
    print(f"n={n}, k={k}, maxPts={m}")
    if k == 0 or n >= k + m - 1:
        print("ca đặc biệt -> 1.0")
        return 1.0
    f = [0.0] * (k + m)
    for x in range(k, n + 1):
        f[x] = 1.0
    s = float(sum(f[k:k + m]))
    print(f"khởi tạo: f[{k}..{n}] = 1, f[{n + 1}..{k + m - 1}] = 0, s = {s}")
    print(f"{'x':>3} {'s trước':>12} {'f[x]=s/m':>12} {'f[x+m]':>7} {'s sau':>12}")
    for x in range(k - 1, -1, -1):
        s_before = s
        f[x] = s / m
        s += f[x] - f[x + m]
        print(f"{x:>3} {s_before:>12.6f} {f[x]:>12.6f} {f[x + m]:>7.3f} {s:>12.6f}")
    print("=> f[0] =", f[0])
    return f[0]


r = trace_window(21, 17, 10)
assert abs(r - 0.73278) < 1e-5

n=21, k=17, maxPts=10
khởi tạo: f[17..21] = 1, f[22..26] = 0, s = 5.0
  x      s trước     f[x]=s/m  f[x+m]        s sau
 16     5.000000     0.500000   0.000     5.500000
 15     5.500000     0.550000   0.000     6.050000
 14     6.050000     0.605000   0.000     6.655000
 13     6.655000     0.665500   0.000     7.320500
 12     7.320500     0.732050   0.000     8.052550
 11     8.052550     0.805255   1.000     7.857805
 10     7.857805     0.785780   1.000     7.643585
  9     7.643585     0.764359   1.000     7.407944
  8     7.407944     0.740794   1.000     7.148738
  7     7.148738     0.714874   1.000     6.863612
  6     6.863612     0.686361   0.500     7.049974
  5     7.049974     0.704997   0.550     7.204971
  4     7.204971     0.720497   0.605     7.320468
  3     7.320468     0.732047   0.665     7.387015
  2     7.387015     0.738701   0.732     7.393666
  1     7.393666     0.739367   0.805     7.327778
  0     7.327778     0.732778   0.786     7.274775
=> f[0] = 0.

**Trace Cách 4 (xuôi)** cho `n = 2, k = 2, maxPts = 2` (ví dụ tay): `top = 3`. `p(0) = 1`. `i = 1`: `p(1) = window/2 = 0,5`; `1 < k` nên `window += 0,5 → 1,5`. `i = 2`: `p(2) = 1,5/2 = 0,75`; `2 ≥ k` và `2 ≤ n`: `ans += 0,75`. `i = 3`: `p(3) = 0,25` (`window` đã bớt `p(1)`? ...). Hàm trace in từng bước:

In [14]:
def trace_forward(n, k, m):
    print(f"n={n}, k={k}, maxPts={m}")
    top = k + m - 1
    p = [0.0] * (top + 1)
    p[0] = 1.0
    window, ans = 1.0, 0.0
    for i in range(1, top + 1):
        p[i] = window / m
        note = ""
        if i < k:
            window += p[i]
            note = f"i<k: còn bốc, cửa sổ += p[{i}]"
        elif i <= n:
            ans += p[i]
            note = f"kết thúc tại {i} <= n: ans += p[{i}]"
        else:
            note = f"kết thúc tại {i} > n: không cộng"
        if 0 <= i - m < k:
            window -= p[i - m]
            note += f"; bớt p[{i - m}] khỏi cửa sổ"
        print(f"i={i:2d}: p[{i}] = {p[i]:.6f} | {note} | window = {window:.6f} | ans = {ans:.6f}")
    print("=> đáp án", ans)
    return ans


assert abs(trace_forward(2, 2, 2) - 0.75) < 1e-12
print()
r = trace_forward(21, 17, 10)
assert abs(r - 0.73278) < 1e-5

n=2, k=2, maxPts=2
i= 1: p[1] = 0.500000 | i<k: còn bốc, cửa sổ += p[1] | window = 1.500000 | ans = 0.000000
i= 2: p[2] = 0.750000 | kết thúc tại 2 <= n: ans += p[2]; bớt p[0] khỏi cửa sổ | window = 0.500000 | ans = 0.750000
i= 3: p[3] = 0.250000 | kết thúc tại 3 > n: không cộng; bớt p[1] khỏi cửa sổ | window = 0.000000 | ans = 0.750000
=> đáp án 0.75

n=21, k=17, maxPts=10
i= 1: p[1] = 0.100000 | i<k: còn bốc, cửa sổ += p[1] | window = 1.100000 | ans = 0.000000
i= 2: p[2] = 0.110000 | i<k: còn bốc, cửa sổ += p[2] | window = 1.210000 | ans = 0.000000
i= 3: p[3] = 0.121000 | i<k: còn bốc, cửa sổ += p[3] | window = 1.331000 | ans = 0.000000
i= 4: p[4] = 0.133100 | i<k: còn bốc, cửa sổ += p[4] | window = 1.464100 | ans = 0.000000
i= 5: p[5] = 0.146410 | i<k: còn bốc, cửa sổ += p[5] | window = 1.610510 | ans = 0.000000
i= 6: p[6] = 0.161051 | i<k: còn bốc, cửa sổ += p[6] | window = 1.771561 | ans = 0.000000
i= 7: p[7] = 0.177156 | i<k: còn bốc, cửa sổ += p[7] | window = 1.948717 | ans = 0.

## 9. Code nộp LeetCode

Chọn **DP tổng trượt** (`O(k + maxPts)`). Giữ nguyên `class Solution`, `new21Game(self, n, k, maxPts)`:

In [15]:
from typing import List


class Solution:
    def new21Game(self, n: int, k: int, maxPts: int) -> float:
        # Không bốc (k = 0) hoặc điểm tối đa k-1+maxPts <= n: chắc chắn thành công
        if k == 0 or n >= k + maxPts - 1:
            return 1.0
        f = [0.0] * (k + maxPts)           # f[x]: xác suất cuối cùng <= n khi đang có x điểm
        for x in range(k, n + 1):          # đã dừng và <= n
            f[x] = 1.0
        s = float(sum(f[k:k + maxPts]))    # tổng cửa sổ f[x+1..x+maxPts] cho x = k-1
        for x in range(k - 1, -1, -1):
            f[x] = s / maxPts
            s += f[x] - f[x + maxPts]      # trượt cửa sổ: thêm f[x], bớt f[x + maxPts]
        return f[0]


print(Solution().new21Game(10, 1, 10), Solution().new21Game(6, 1, 10), round(Solution().new21Game(21, 17, 10), 5))
assert Solution().new21Game(10, 1, 10) == 1.0
assert abs(Solution().new21Game(6, 1, 10) - 0.6) < 1e-9
assert abs(Solution().new21Game(21, 17, 10) - 0.73278) < 1e-5

1.0 0.6 0.73278


## 10. Vì sao đúng

**(a) Truy hồi.** Theo luật xác suất toàn phần trên kết quả lần bốc đầu: với `x < k`, `f(x) = Σ_{i=1}^{m} (1/m) f(x + i)`. Với `x ≥ k` đã dừng: `f(x) = 1` nếu `x ≤ n`, `0` nếu khác. Quá trình dừng vì mỗi lần bốc tăng điểm ít nhất `1` và điểm luôn `< k` trước mỗi lần bốc; nên sau tối đa `k` lần bốc đã dừng.

**(b) Tổng trượt.** `S(x) = Σ_{i=1}^{m} f(x + i)`. Khi chuyển từ `x` sang `x − 1`: `S(x − 1) = Σ_{i=1}^{m} f(x − 1 + i) = f(x) + Σ_{i=2}^{m} f(x − 1 + i) = f(x) + (S(x) − f(x + m))`. Vì `x − 1 + i` chạy `x, …, x + m − 1`, còn `S(x)` chạy `x + 1, …, x + m`: chênh nhau đúng ở hai đầu. Do đó `S(x − 1) = S(x) + f(x) − f(x + m)`. Khởi đầu `S(k − 1) = f(k) + … + f(k + m − 1)`.

**(c) Ca đặc biệt.** Nếu `k = 0` Alice không bốc: điểm `0 ≤ n` (vì `0 ≤ k ≤ n`) nên xác suất `1`. Điểm cuối cùng lớn nhất là `(k − 1) + m` (bốc `m` từ `k − 1`); nếu `n ≥ k + m − 1` thì mọi kết cục `≤ n`: xác suất `1` (và `f` bảng cũng cho `1`, nhưng cắt ngắn tránh truy cập ngoài mảng).

**(d) Cách xuôi cho cùng đáp án.** `p(i)` là xác suất đi qua `i`; vì từ mỗi điểm `j < k` đến `i` có đúng một giá trị bốc `i − j` (nếu `1 ≤ i − j ≤ m`) xác suất `1/m`, `p(i) = (1/m) Σ p(j)`. Mỗi trò kết thúc tại **đúng một** điểm `i ≥ k` (điểm đầu tiên `≥ k`), nên các biến cố "kết thúc tại `i`" là rời nhau và `P(kết thúc tại i) = p(i)` (vì tại `i ≥ k` ta không bốc tiếp nên `i` không được dùng làm điểm trung gian). Cộng các `i ≤ n`.

**(e) Sai số số thực.** Cách 3 dùng cộng/trừ liên tiếp; mỗi bước sai số tương đối `~10^-16`, tích luỹ qua `k ≤ 10^4` bước thì cỡ `10^-12`, nhỏ hơn dung sai `10^-5` nhiều bậc. Ta kiểm chứng bằng `Fraction` (mục 12D).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Đệ quy thuần | `O(maxPts^k)` | `O(k)` | oracle cho input rất nhỏ |
| 2. DP cộng đủ `maxPts` số | `O(k · maxPts)` | `O(k + maxPts)` | `10^8` ở giới hạn: hàng giây hoặc quá chậm |
| 3. DP tổng trượt | `O(k + maxPts)` | `O(k + maxPts)` | **đáp án chính** |
| 4. DP xuôi `p(i)` | `O(k + maxPts)` | `O(k + maxPts)` | cho cả phân phối điểm cuối |

Đo thực tế (`n` lớn tới `10^4`):

In [16]:
def bench(fn, args, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn(*args)
        best = min(best, time.perf_counter() - t0)
    return best


for n, k, m in [(10 ** 4, 10 ** 4, 10 ** 4), (9000, 5000, 5000), (3000, 2000, 2000)]:
    ref = game_window(n, k, m)
    row = [f"window {bench(game_window, (n, k, m)) * 1000:8.2f} ms", f"forward {bench(game_forward, (n, k, m)) * 1000:8.2f} ms"]
    if k * m <= 4 * 10 ** 6:
        row.append(f"dp(sum) {bench(game_dp, (n, k, m), 1) * 1000:9.1f} ms")
    assert abs(game_forward(n, k, m) - ref) < 1e-9
    print(f"n={n:5d} k={k:5d} maxPts={m:5d}: đáp án {ref:.6f} | " + " | ".join(row))
t = bench(game_dp, (4000, 2000, 2000), 1)
print(f"dp(sum) k=maxPts=2000: {t * 1000:.0f} ms -> k=maxPts=10^4 khoảng {t * 25:.1f} s (tăng theo k·maxPts)")

n=10000 k=10000 maxPts=10000: đáp án 0.000272 | window     0.83 ms | forward     2.12 ms
n= 9000 k= 5000 maxPts= 5000: đáp án 0.949154 | window     0.50 ms | forward     1.19 ms
n= 3000 k= 2000 maxPts= 2000: đáp án 0.710965 | window     0.18 ms | forward     0.42 ms | dp(sum)      35.9 ms


dp(sum) k=maxPts=2000: 35 ms -> k=maxPts=10^4 khoảng 0.9 s (tăng theo k·maxPts)


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **so sánh chính xác bằng `Fraction`** (và dung sai cho `float`), (E) mô phỏng Monte Carlo với RNG tự dựng có seed, (F) giới hạn lớn nhất.

Quy ước: so sánh `float` bằng `abs(a − b) <= 1e-9` giữa các cài đặt và `<= 1e-5` (dung sai của đề) so với đáp án chính xác.

### A. Ví dụ của đề

In [17]:
IMPLS = {
    "dp": game_dp,
    "window": game_window,
    "forward": game_forward,
    "Solution": lambda n, k, m: Solution().new21Game(n, k, m),
    "SolutionDP": lambda n, k, m: SolutionDP().new21Game(n, k, m),
    "SolutionWindow": lambda n, k, m: SolutionWindow().new21Game(n, k, m),
    "SolutionForward": lambda n, k, m: SolutionForward().new21Game(n, k, m),
}
EPS = 1e-9


def close(a, b, eps=EPS):
    return abs(a - b) <= eps


def check(n, k, m, want, eps=EPS, names=None):
    for name in (names or IMPLS):
        got = IMPLS[name](n, k, m)
        assert close(got, float(want), eps), (name, (n, k, m), got, float(want))


check(10, 1, 10, 1)
check(6, 1, 10, Fraction(3, 5))
check(21, 17, 10, f_values(21, 17, 10)[0])
assert abs(Solution().new21Game(21, 17, 10) - 0.73278) < 1e-5
assert game_brute(21, 3, 3) == f_values(21, 3, 3)[0]
print("ví dụ 1, 2, 3: cả", len(IMPLS), "cài đặt đúng (ví dụ 3 so với Fraction chính xác =", float(f_values(21, 17, 10)[0]), ")")

ví dụ 1, 2, 3: cả 7 cài đặt đúng (ví dụ 3 so với Fraction chính xác = 0.7327777870686081 )


### B. Edge case

- `k = 0` (không bốc) → `1`, kể cả `n = 0`.
- `n = k` (phải dừng ngay đúng `k`).
- `maxPts = 1` (mỗi lần được `1` điểm): điểm cuối đúng `k` → xác suất `1` nếu `n ≥ k`.
- `n ≥ k + maxPts − 1` → `1`; `n = k + maxPts − 2` (ngay dưới ngưỡng) **không** là `1`.
- `n = k` với `maxPts` lớn.
- `k = 1`: bốc một lần → `min(n, maxPts) / maxPts`.

In [18]:
# k = 0
for n, m in [(0, 5), (7, 1), (10 ** 4, 10 ** 4)]:
    check(n, 0, m, 1)
# maxPts = 1
for n, k in [(5, 5), (9, 4), (10, 10)]:
    check(n, k, 1, 1)
# k = 1: min(n, m) / m
for n, m in [(1, 10), (5, 10), (10, 10), (3, 3), (2, 7)]:
    check(n, 1, m, Fraction(min(n, m), m))
# sát ngưỡng n = k + m - 1
for k, m in [(3, 4), (10, 6), (17, 10)]:
    check(k + m - 1, k, m, 1)
    just_below = f_values(k + m - 2, k, m)[0]
    assert just_below < 1
    check(k + m - 2, k, m, just_below)
# n = k
for k, m in [(1, 5), (4, 4), (6, 3)]:
    check(k, k, m, f_values(k, k, m)[0])
print("edge case: tất cả cài đặt đúng")

edge case: tất cả cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_boundary_k(n, k, m):                      # f(x) = 1 chỉ cho k < x <= n (bỏ x = k): coi x == k là còn bốc
    f = [0.0] * (k + m + 1)
    for x in range(k + 1, min(n, k + m) + 1):
        f[x] = 1.0
    for x in range(k - 1, -1, -1):
        f[x] = sum(f[x + 1:x + m + 1]) / m
    return f[0]


def bug_no_divide(n, k, m):                       # quên chia maxPts
    f = [0.0] * (k + m)
    for x in range(k, min(n, k + m - 1) + 1):
        f[x] = 1.0
    for x in range(k - 1, -1, -1):
        f[x] = sum(f[x + 1:x + m + 1])
    return f[0]


def bug_window_off_by_one(n, k, m):               # cửa sổ lệch một: bớt f[x+m-1] thay f[x+m]
    if k == 0 or n >= k + m - 1:
        return 1.0
    f = [0.0] * (k + m)
    for x in range(k, n + 1):
        f[x] = 1.0
    s = float(sum(f[k:k + m]))
    for x in range(k - 1, -1, -1):
        f[x] = s / m
        s += f[x] - f[x + m - 1]
    return f[0]


def bug_early_exit(n, k, m):                      # ngưỡng thoát sớm sai: n >= k + m - 2
    if k == 0 or n >= k + m - 2:
        return 1.0
    return game_window(n, k, m)


def bug_k_plus_one(n, k, m):                      # coi dừng khi > k thay vì >= k
    f = [0.0] * (k + 1 + m)
    for x in range(k + 1, min(n, k + m) + 1):
        f[x] = 1.0
    for x in range(k, -1, -1):
        f[x] = sum(f[x + 1:x + m + 1]) / m
    return f[0]


BUGS = [bug_boundary_k, bug_no_divide, bug_window_off_by_one, bug_early_exit, bug_k_plus_one]
rng = random.Random(1)
tests = []
for _ in range(500):
    m = rng.randint(1, 8)
    k = rng.randint(0, 12)
    n = rng.randint(k, k + 12)
    tests.append((n, k, m))
for bug in BUGS:
    bad = next((t for t in tests if abs(bug(*t) - float(f_values(*t)[0])) > 1e-6), None)
    assert bad is not None, bug.__name__
    want = float(f_values(*bad)[0])
    print(f"{bug.__name__:22s} sai tại (n,k,m) = {bad}: ra {bug(*bad):.6f}, đúng {want:.6f}")

bug_boundary_k         sai tại (n,k,m) = (21, 9, 3): ra 0.497587, đúng 1.000000
bug_no_divide          sai tại (n,k,m) = (21, 9, 3): ra 355.000000, đúng 1.000000
bug_window_off_by_one  sai tại (n,k,m) = (16, 10, 8): ra 0.674343, đúng 0.975535
bug_early_exit         sai tại (n,k,m) = (16, 10, 8): ra 1.000000, đúng 0.975535
bug_k_plus_one         sai tại (n,k,m) = (5, 4, 2): ra 0.656250, đúng 1.000000


### D. So sánh chính xác bằng `Fraction`

**D1.** Đệ quy thuần `game_brute` (chính xác) bằng bảng `f_values` (chính xác) trên mọi `(n, k, m)` nhỏ: `m ∈ 1..3`, `k ∈ 0..6`, `n ∈ k..k + 8` (kiểm tra bảng).

**D2.** Mọi cài đặt `float` khớp `Fraction` trong `1e-9` trên lưới tham số `k ∈ 0..40`, `m ∈ 1..30`, `n ∈ k..k + m + 2`.

**D3.** Một số ca lớn hơn (`k` tới `300`, `m` tới `100`): `Fraction` chính xác bằng cách tổng trượt **số hữu tỉ** (bản `Fraction` của Cách 3) so với `float`.

In [20]:
cnt = 0
for m in range(1, 4):
    for k in range(0, 7):
        for n in range(k, k + 9):
            assert game_brute(n, k, m) == f_values(n, k, m)[0], (n, k, m)
            cnt += 1
print("D1:", cnt, "bộ (n,k,m): đệ quy thuần == bảng f_values (Fraction, bằng nhau tuyệt đối)")

t0 = time.perf_counter()
cnt = 0
worst = 0.0
for k in range(0, 41):
    for m in range(1, 31):
        for n in sorted({k, k + m // 2, k + m - 2, k + m - 1, k + m + 2, k + 1, k + m // 3}):
            if n < k:
                continue
            want = float(f_values(n, k, m)[0])
            for name in IMPLS:
                got = IMPLS[name](n, k, m)
                worst = max(worst, abs(got - want))
                assert close(got, want), (name, n, k, m, got, want)
            cnt += 1
print(f"D2: {cnt} bộ (n,k,m) trên lưới, 7 cài đặt khớp Fraction; sai số float lớn nhất {worst:.2e}; mất {time.perf_counter() - t0:.1f} s")
assert worst < 1e-12

D1: 189 bộ (n,k,m): đệ quy thuần == bảng f_values (Fraction, bằng nhau tuyệt đối)


D2: 8036 bộ (n,k,m) trên lưới, 7 cài đặt khớp Fraction; sai số float lớn nhất 4.22e-15; mất 2.5 s


In [21]:
def game_window_exact(n, k, m):
    """Cách 3 với Fraction: tổng trượt chính xác (không sai số)."""
    if k == 0 or n >= k + m - 1:
        return Fraction(1)
    f = [Fraction(0)] * (k + m)
    for x in range(k, n + 1):
        f[x] = Fraction(1)
    s = sum(f[k:k + m])
    for x in range(k - 1, -1, -1):
        f[x] = s / m
        s += f[x] - f[x + m]
    return f[0]


t0 = time.perf_counter()
rng = random.Random(837)
worst = 0.0
for _ in range(40):
    k = rng.randint(1, 300)
    m = rng.randint(1, 100)
    n = rng.randint(k, k + m + 5)
    ex = game_window_exact(n, k, m)
    assert ex == f_values(n, k, m)[0] if k + m <= 150 else True                      # đối chiếu bảng khi đủ nhỏ
    for name in ("window", "forward", "Solution", "SolutionForward"):
        worst = max(worst, abs(IMPLS[name](n, k, m) - float(ex)))
        assert close(IMPLS[name](n, k, m), float(ex)), (name, n, k, m)
print(f"D3: 40 bộ lớn (k <= 300, m <= 100): float khớp Fraction chính xác, sai số lớn nhất {worst:.2e} ({time.perf_counter() - t0:.1f} s)")
assert worst < 1e-12

D3: 40 bộ lớn (k <= 300, m <= 100): float khớp Fraction chính xác, sai số lớn nhất 1.01e-13 (0.1 s)


### E. Mô phỏng Monte Carlo (RNG tự dựng có seed)

Mô phỏng trực tiếp trò chơi bằng bộ sinh số tự dựng `XorShift64` (cùng seed → cùng kết quả), lấy mẫu loại bỏ để chọn đều trong `[1, maxPts]`. Mỗi ván cho `1` (điểm cuối `≤ n`) hoặc `0`; trung bình `T` ván hội tụ về đáp án với sai số chuẩn `σ ≤ 0,5/√T`. Ta yêu cầu `|trung bình − đáp án chính xác| ≤ 4σ`.

In [22]:
class XorShift64:
    MASK = (1 << 64) - 1

    def __init__(self, seed):
        z = (seed + 0x9E3779B97F4A7C15) & self.MASK
        z = ((z ^ (z >> 30)) * 0xBF58476D1CE4E5B9) & self.MASK
        z = ((z ^ (z >> 27)) * 0x94D049BB133111EB) & self.MASK
        z ^= z >> 31
        self.state = z or 0x2545F4914F6CDD1D

    def next_u64(self):
        x = self.state
        x ^= (x << 13) & self.MASK
        x ^= x >> 7
        x ^= (x << 17) & self.MASK
        self.state = x
        return x

    def randint(self, a, b):
        n = b - a + 1
        limit = (1 << 64) - ((1 << 64) % n)
        while True:
            r = self.next_u64()
            if r < limit:
                return a + r % n


def simulate(n, k, m, trials, seed):
    rng = XorShift64(seed)
    good = 0
    for _ in range(trials):
        pts = 0
        while pts < k:
            pts += rng.randint(1, m)
        good += pts <= n
    return good / trials


T = 60000
sigma = 0.5 / math.sqrt(T)
print(f"T = {T}, σ <= {sigma:.5f}, ngưỡng 4σ = {4 * sigma:.5f}")
for (n, k, m), seed in [((21, 17, 10), 1), ((6, 1, 10), 2), ((22, 20, 6), 3), ((50, 40, 20), 4), ((100, 90, 12), 5), ((10, 8, 4), 6)]:
    exact = Solution().new21Game(n, k, m)
    est = simulate(n, k, m, T, seed)
    print(f"(n,k,m) = {(n, k, m)!s:>14}: đáp án {exact:.5f}, mô phỏng {est:.5f}, z = {(est - exact) / sigma:+.2f}")
    assert abs(est - exact) <= 4 * sigma, (n, k, m)
assert simulate(21, 17, 10, 2000, 42) == simulate(21, 17, 10, 2000, 42)
print("mô phỏng khớp trong 4σ; cùng seed cho cùng kết quả")

T = 60000, σ <= 0.00204, ngưỡng 4σ = 0.00816
(n,k,m) =   (21, 17, 10): đáp án 0.73278, mô phỏng 0.73357, z = +0.39


(n,k,m) =     (6, 1, 10): đáp án 0.60000, mô phỏng 0.60182, z = +0.89


(n,k,m) =    (22, 20, 6): đáp án 0.71466, mô phỏng 0.71345, z = -0.59


(n,k,m) =   (50, 40, 20): đáp án 0.78193, mô phỏng 0.78047, z = -0.72


(n,k,m) =  (100, 90, 12): đáp án 0.98718, mô phỏng 0.98742, z = +0.12
(n,k,m) =     (10, 8, 4): đáp án 0.89830, mô phỏng 0.89752, z = -0.38
mô phỏng khớp trong 4σ; cùng seed cho cùng kết quả


Lưu ý: kiểm tra thống kê chỉ cho độ chính xác cỡ `10^-2`; độ chính xác `10^-5` của đề đến từ `Fraction` ở nhóm D.

### F. Giới hạn lớn nhất

`n = k = maxPts = 10^4`; `n = 10^4, k = 5000, maxPts = 10^4`; `k = 10^4, maxPts = 1`; `n = k + maxPts − 2` ngay dưới ngưỡng.

In [23]:
big_cases = [(10 ** 4, 10 ** 4, 10 ** 4), (10 ** 4, 5000, 10 ** 4), (10 ** 4, 10 ** 4, 1), (10 ** 4, 9000, 5000),
             (9999, 5000, 5001), (5000, 5000, 3), (7000, 4000, 4000), (10 ** 4, 1, 10 ** 4)]
fast = ["window", "forward", "Solution", "SolutionForward", "SolutionWindow"]
for n, k, m in big_cases:
    t0 = time.perf_counter()
    vals = [IMPLS[name](n, k, m) for name in fast]
    dt = time.perf_counter() - t0
    assert max(vals) - min(vals) <= 1e-9, (n, k, m, vals)
    assert -1e-12 <= vals[0] <= 1 + 1e-12
    print(f"(n,k,m) = {(n, k, m)!s:>20}: {vals[0]:.8f} (5 cài đặt khớp trong 1e-9; {dt * 1000:.0f} ms)")
assert abs(Solution().new21Game(10 ** 4, 1, 10 ** 4) - 1.0) < 1e-12

(n,k,m) = (10000, 10000, 10000): 0.00027179 (5 cài đặt khớp trong 1e-9; 7 ms)
(n,k,m) = (10000, 5000, 10000): 0.82442245 (5 cài đặt khớp trong 1e-9; 5 ms)
(n,k,m) =    (10000, 10000, 1): 1.00000000 (5 cài đặt khớp trong 1e-9; 3 ms)
(n,k,m) =  (10000, 9000, 5000): 0.36140468 (5 cài đặt khớp trong 1e-9; 5 ms)
(n,k,m) =   (9999, 5000, 5001): 0.99999989 (5 cài đặt khớp trong 1e-9; 4 ms)
(n,k,m) =      (5000, 5000, 3): 0.50000000 (5 cài đặt khớp trong 1e-9; 2 ms)
(n,k,m) =   (7000, 4000, 4000): 0.92182483 (5 cài đặt khớp trong 1e-9; 3 ms)
(n,k,m) =    (10000, 1, 10000): 1.00000000 (5 cài đặt khớp trong 1e-9; 2 ms)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sai biên tại `x = k`

Alice dừng khi có **`k` hoặc nhiều hơn**. Nếu coi `x = k` là "còn bốc" (chỉ dừng khi `> k`), ta tính sai.

In [24]:
print("(n,k,m) = (21,17,10): bug_k_plus_one =", round(bug_k_plus_one(21, 17, 10), 5), "| bug_boundary_k =", round(bug_boundary_k(21, 17, 10), 5), "| đúng:", round(Solution().new21Game(21, 17, 10), 5))
assert abs(bug_k_plus_one(21, 17, 10) - Solution().new21Game(21, 17, 10)) > 1e-3
assert abs(bug_boundary_k(21, 17, 10) - Solution().new21Game(21, 17, 10)) > 1e-3

(n,k,m) = (21,17,10): bug_k_plus_one = 0.62135 | bug_boundary_k = 0.54707 | đúng: 0.73278


**Vì sao sai:** ở điểm đúng `k` Alice không bốc nữa. **Đúng:** `f(x) = 1 nếu k ≤ x ≤ n` và điền `f[x] = 1` cho `x ∈ [k, n]`.

### Lỗi 2: quên chia `maxPts`

`f(x)` là **trung bình** của `maxPts` giá trị `f(x + i)`, không phải tổng. Quên chia cho giá trị lớn hơn `1` (không phải xác suất).

In [25]:
print("quên chia:", bug_no_divide(6, 1, 10), "| đúng:", Solution().new21Game(6, 1, 10))
assert bug_no_divide(6, 1, 10) == 6.0

quên chia: 6.0 | đúng: 0.6


**Vì sao sai:** mỗi lần bốc có xác suất `1/maxPts`. **Đúng:** `f[x] = s / maxPts`.

### Lỗi 3: cửa sổ trượt lệch một

`S(x − 1) = S(x) + f(x) − f(x + maxPts)`. Bớt `f(x + maxPts − 1)` (hay `f(x + maxPts + 1)`) làm tổng sai và sai số lan nhanh.

In [26]:
n, k, m = 21, 17, 10
print("cửa sổ lệch:", round(bug_window_off_by_one(n, k, m), 5), "| đúng:", round(Solution().new21Game(n, k, m), 5))
assert abs(bug_window_off_by_one(n, k, m) - Solution().new21Game(n, k, m)) > 1e-3

cửa sổ lệch: 0.51973 | đúng: 0.73278


**Vì sao sai:** phần tử rời cửa sổ là `f[x + maxPts]` (đầu xa), không phải `f[x + maxPts − 1]` (còn trong cửa sổ). **Đúng:** `s += f[x] − f[x + maxPts]`; kiểm tra bằng cách so với DP cộng đủ `maxPts` số.

### Lỗi 4: ngưỡng thoát sớm sai

`n ≥ k + maxPts − 1` là điều kiện **đúng** (điểm tối đa `k − 1 + maxPts`). Viết `k + maxPts − 2` trả `1` cho cả những ca xác suất `< 1`.

In [27]:
n, k, m = 3 + 4 - 2, 3, 4          # n = k + m - 2: ngay dưới ngưỡng
print("n =", n, "| thoát sớm:", bug_early_exit(n, k, m), "| đúng:", Solution().new21Game(n, k, m))
assert bug_early_exit(n, k, m) == 1.0 and Solution().new21Game(n, k, m) < 1

n = 5 | thoát sớm: 1.0 | đúng: 0.921875


**Vì sao sai:** với `n = k + m − 2`, kết cục `k + m − 1` (bốc `m` từ `k − 1`) vượt `n`. **Đúng:** `n >= k + maxPts - 1`.

### Lỗi 5: đệ quy có nhớ quá sâu và quá chậm

`@cache` với `f(x)` sâu `k` tầng cho `k = 10^4` gây `RecursionError`; cộng đủ `maxPts` số mỗi `x` là `O(k · maxPts) = 10^8`.

In [28]:
def game_memo(n, k, m):
    @cache
    def f(x):
        if x >= k:
            return 1.0 if x <= n else 0.0
        return sum(f(x + i) for i in range(1, m + 1)) / m

    return f(0)


old = sys.getrecursionlimit()
sys.setrecursionlimit(1000)
try:
    game_memo(10 ** 4, 5000, 3000)
    print("không lỗi (bất ngờ)")
except RecursionError:
    print("game_memo(10^4, 5000, 3000) -> RecursionError (độ sâu ~ k)")
finally:
    sys.setrecursionlimit(old)
print("DP tổng trượt cùng input:", Solution().new21Game(10 ** 4, 5000, 3000))

game_memo(10^4, 5000, 3000) -> RecursionError (độ sâu ~ k)
DP tổng trượt cùng input: 1.0


**Vì sao sai:** độ sâu và thời gian phụ thuộc `k`, `maxPts` lớn. **Đúng:** DP bảng + tổng trượt.

### Lỗi 6 (bonus): quên các ca đặc biệt `k = 0`

Với `k = 0` Alice không bốc; một số cài đặt vẫn "bốc một lần". Kết quả đúng là `1.0`.

In [29]:
print("k=0:", Solution().new21Game(0, 0, 5), Solution().new21Game(5, 0, 1), "(cả hai phải là 1.0)")
assert Solution().new21Game(0, 0, 5) == 1.0

k=0: 1.0 1.0 (cả hai phải là 1.0)


## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "xác suất kết quả của quá trình bốc/gieo lặp cho tới khi vượt ngưỡng", "mỗi bước cộng giá trị đều trong `[1, m]`" → **DP xác suất** với **tổng trượt** (cửa sổ độ rộng `m`).

**Mẹo:**

1. Định nghĩa `f(x)` (ngược) hoặc `p(i)` (xuôi) rõ ràng; kiểm tra bằng tay ví dụ nhỏ.
2. Khi truy hồi là **trung bình của `m` ô liên tiếp** → tổng trượt `O(1)` mỗi ô (thêm mới, bớt cũ).
3. Xác định **ca đặc biệt** cho đáp án ngay (`k = 0`, `n ≥ k + m − 1`).
4. Dùng `Fraction` làm oracle, `float` có dung sai trong test; mô phỏng seed để kiểm thống kê.
5. Hai chiều tính (ngược `f`, xuôi `p`) cho cùng đáp án — cross-check.
6. Tránh đệ quy sâu khi tham số lên `10^4`.

**Họ bài cùng pattern:** Soup Servings (808), Knight Probability in Chessboard (688), Dice Roll Simulation (1223), Number of Dice Rolls With Target Sum (1155), Domino and Tromino Tiling (790).

## 15. Biến thể

1. **Phân phối điểm cuối** (bài tập 5): `P(điểm cuối = s)` với `s ∈ [k, k + m − 1]`.
2. **Xác suất điểm cuối `> n`** (phần bù) hoặc **trong khoảng `[a, b]`**.
3. **Kỳ vọng số lần bốc** để dừng.
4. **Xúc xắc không đều**: xác suất khác nhau cho mỗi mặt (tổng trượt có trọng số không còn dùng được; cần nhân chập).
5. **`k` điểm nhưng cho phép "dừng sớm"** (chiến lược tối ưu): thành bài tối ưu hoá (bài Blackjack).

Cài đặt biến thể 2 (khoảng) và 3 (kỳ vọng số lần bốc) bằng `Fraction`, đối chiếu mô phỏng:

In [30]:
def final_distribution(k, m):
    """Biến thể 1: {s: P(điểm cuối = s)} cho s trong [k, k+m-1] (Fraction chính xác); k = 0 -> {0: 1}."""
    if k == 0:
        return {0: Fraction(1)}
    top = k + m - 1
    p = [Fraction(0)] * (top + 1)
    p[0] = Fraction(1)
    window = Fraction(1)
    for i in range(1, top + 1):
        p[i] = window / m
        if i < k:
            window += p[i]
        if 0 <= i - m < k:
            window -= p[i - m]
    return {s: p[s] for s in range(k, top + 1)}


def expected_draws(k, m):
    """Biến thể 3: kỳ vọng số lần bốc (Fraction). E(x) = 1 + trung bình E(x+i) trên các x+i < k; E(x >= k) = 0."""
    E = {}
    for x in range(k - 1, -1, -1):
        E[x] = 1 + sum(E.get(x + i, Fraction(0)) for i in range(1, m + 1) if x + i < k) / m
    return E.get(0, Fraction(0))


dist = final_distribution(17, 10)
print("tổng xác suất:", sum(dist.values()), "| P(điểm cuối <= 21) =", float(sum(v for s, v in dist.items() if s <= 21)))
assert sum(dist.values()) == 1
assert sum(v for s, v in dist.items() if s <= 21) == f_values(21, 17, 10)[0]
print("E[số lần bốc] k=17, m=10:", float(expected_draws(17, 10)))
assert expected_draws(1, 10) == 1 and expected_draws(0, 5) == 0
rng = XorShift64(9)
total = 0
for _ in range(30000):
    pts = draws = 0
    while pts < 17:
        pts += rng.randint(1, 10)
        draws += 1
    total += draws
print("mô phỏng kỳ vọng:", total / 30000)
assert abs(total / 30000 - float(expected_draws(17, 10))) < 0.05

tổng xác suất: 1 | P(điểm cuối <= 21) = 0.7327777870686081
E[số lần bốc] k=17, m=10: 3.6286669863572163


mô phỏng kỳ vọng: 3.6336333333333335


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Đệ quy thuần | `f(x) = trung bình f(x+i)` không nhớ | `O(maxPts^k)` | `O(k)` | Oracle, input rất nhỏ |
| 2. DP cộng đủ | Điền `f` giảm dần, mỗi ô cộng `maxPts` số | `O(k · maxPts)` | `O(k + maxPts)` | Hiểu bản chất; chậm ở giới hạn |
| 3. DP tổng trượt | `S(x−1) = S(x) + f(x) − f(x+maxPts)` | `O(k + maxPts)` | `O(k + maxPts)` | **Đáp án chính** |
| 4. DP xuôi `p(i)` | Xác suất đi qua điểm, cộng các điểm cuối `≤ n` | `O(k + maxPts)` | `O(k + maxPts)` | Cần cả phân phối điểm cuối |

Cả bốn khớp `Fraction` chính xác trên hơn 1000 bộ tham số (mục 12D) và khớp mô phỏng Monte Carlo trong 4σ (mục 12E).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [31]:
print(Solution().new21Game(0, 0, 7), Solution().new21Game(4, 1, 4), Solution().new21Game(3, 1, 4), Solution().new21Game(100, 3, 5))
print([x for x in range(3, 3 + 5)], "<- các điểm cuối có thể khi k=3, maxPts=5 là 3..7")

1.0 1.0 0.75 1.0
[3, 4, 5, 6, 7] <- các điểm cuối có thể khi k=3, maxPts=5 là 3..7


### Bài 2 (tính tay)

- (a) `n = 2, k = 2, maxPts = 2`: tính `f(3), f(2), f(1), f(0)`.
- (b) `n = 3, k = 2, maxPts = 3`: điểm cuối nằm trong `[2, 4]`; tính xác suất `≤ 3`.
- (c) Dùng tổng trượt cho (b): `s` ban đầu và các bước.

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện DP tổng trượt:

In [32]:
def window_todo(n, k, m):
    if k == 0 or n >= k + m - 1:
        return 1.0
    f = [0.0] * (k + m)
    for x in range(k, n + 1):
        f[x] = 1.0
    s = float(sum(f[k:k + m]))
    for x in range(k - 1, -1, -1):
        # TODO 1: f[x] = trung bình cửa sổ
        # TODO 2: cập nhật s cho x - 1 (thêm f[x], bớt f[x + m])
        pass
    return f[0]


# Khi làm xong, bỏ comment để kiểm tra:
# assert abs(window_todo(21, 17, 10) - 0.73278) < 1e-5 and abs(window_todo(2, 2, 2) - 0.75) < 1e-12

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng DP xuôi `p(i)` (hoặc tổng trượt):

In [33]:
class MySolution:
    def new21Game(self, n, k, maxPts):
        pass            # TODO


# Khi làm xong:
# assert abs(MySolution().new21Game(6, 1, 10) - 0.6) < 1e-9
# assert abs(MySolution().new21Game(21, 17, 10) - 0.73278) < 1e-5

### Bài 5 (mini challenge): phân phối điểm cuối

Viết `final_scores(k, maxPts)` trả về danh sách xác suất `[P(điểm cuối = k), …, P(điểm cuối = k + maxPts − 1)]` bằng `float`, với tổng `≈ 1` (với `k = 0` trả `[1.0]`). Đáp án của bài gốc là tổng các phần tử có điểm `≤ n`. Kiểm bằng Monte Carlo hoặc so với `final_distribution` (Fraction).

In [34]:
def final_scores(k, maxPts):
    pass            # TODO


# Khi làm xong:
# fs = final_scores(17, 10)
# assert len(fs) == 10 and abs(sum(fs) - 1) < 1e-9 and abs(sum(fs[:5]) - 0.73278) < 1e-5

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: `k = 0` → `1`. `n ≥ k + maxPts − 1` → `1`.
- Mức 2: `(4, 1, 4)`: `n = 4 ≥ 1 + 4 − 1 = 4` → `1`; `(3, 1, 4)`: bốc một lần, `3/4`.
- Mức 3: `1.0, 1.0, 0.75, 1.0` (với `(100, 3, 5)` chắc chắn `≤ n`).

**Bài 2**

- Mức 1: `f(x) = 1` nếu `k ≤ x ≤ n`, `0` nếu `x > n`.
- Mức 2: (a) `f(3) = 0, f(2) = 1, f(1) = ½, f(0) = ¾`.
- Mức 3: (b) `f(4) = 0, f(3) = f(2) = 1`; `f(1) = (f(2) + f(3) + f(4))/3 = 2/3`; `f(0) = (f(1) + f(2) + f(3))/3 = (2/3 + 1 + 1)/3 = 8/9`. (c) `s = f(2) + f(3) + f(4) = 2`; `f(1) = 2/3`; `s ← 2 + 2/3 − f(4) = 8/3`; `f(0) = 8/9`.

**Bài 3**

- Mức 1: `f[x] = s / m`.
- Mức 2: `s += f[x] - f[x + m]`.
- Mức 3: thứ tự: gán `f[x]` rồi mới cập nhật `s`.

**Bài 4**

- Mức 1: `p[0] = 1`, `window = 1`.
- Mức 2: `p[i] = window / m`; nếu `i < k` thêm vào cửa sổ; nếu `i ≥ k` và `i ≤ n` cộng vào đáp án; bớt `p[i − m]` nếu `0 ≤ i − m < k`.
- Mức 3: xem Cách 4 ở mục 7.

**Bài 5**

- Mức 1: chính là `p[k], …, p[k + maxPts − 1]` của DP xuôi.
- Mức 2: `k = 0` đặc biệt.
- Mức 3: kiểm tra tổng bằng `1`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [35]:
assert (Solution().new21Game(0, 0, 7), Solution().new21Game(4, 1, 4), Solution().new21Game(100, 3, 5)) == (1.0, 1.0, 1.0)
assert abs(Solution().new21Game(3, 1, 4) - 0.75) < 1e-12
print("Bài 1 OK")

Bài 1 OK


### Bài 2

(a) `f(3) = 0`, `f(2) = 1`, `f(1) = ½`, `f(0) = ¾`. (b) `8/9`. (c) `s = 2` → `f(1) = 2/3`; `s = 2 + 2/3 − 0 = 8/3` → `f(0) = 8/9`.

In [36]:
f = f_values(2, 2, 2)
assert (f[3], f[2], f[1], f[0]) == (0, 1, Fraction(1, 2), Fraction(3, 4))
f = f_values(3, 2, 3)
assert f[4] == 0 and f[3] == f[2] == 1 and f[1] == Fraction(2, 3) and f[0] == Fraction(8, 9)
s = f[2] + f[3] + f[4]
assert s == 2 and s / 3 == f[1]
s = s + f[1] - f[4]
assert s == Fraction(8, 3) and s / 3 == f[0]
assert game_window_exact(3, 2, 3) == Fraction(8, 9) and game_brute(3, 2, 3) == Fraction(8, 9)
print("Bài 2 OK: (b) = 8/9 =", float(f[0]))

Bài 2 OK: (b) = 8/9 = 0.8888888888888888


### Bài 3

In [37]:
def window_answer(n, k, m):
    if k == 0 or n >= k + m - 1:
        return 1.0
    f = [0.0] * (k + m)
    for x in range(k, n + 1):
        f[x] = 1.0
    s = float(sum(f[k:k + m]))
    for x in range(k - 1, -1, -1):
        f[x] = s / m
        s += f[x] - f[x + m]
    return f[0]


assert abs(window_answer(21, 17, 10) - 0.73278) < 1e-5 and abs(window_answer(2, 2, 2) - 0.75) < 1e-12
rng = random.Random(3)
for _ in range(200):
    k = rng.randint(0, 40); m = rng.randint(1, 20); n = rng.randint(k, k + m + 3)
    assert close(window_answer(n, k, m), float(f_values(n, k, m)[0]))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [38]:
class MySolutionAnswer:
    def new21Game(self, n, k, maxPts):
        if k == 0:
            return 1.0
        top = k + maxPts - 1
        p = [0.0] * (top + 1)
        p[0] = 1.0
        window, ans = 1.0, 0.0
        for i in range(1, top + 1):
            p[i] = window / maxPts
            if i < k:
                window += p[i]
            elif i <= n:
                ans += p[i]
            if 0 <= i - maxPts < k:
                window -= p[i - maxPts]
        return ans


assert abs(MySolutionAnswer().new21Game(6, 1, 10) - 0.6) < 1e-9
assert abs(MySolutionAnswer().new21Game(21, 17, 10) - 0.73278) < 1e-5
rng = random.Random(4)
for _ in range(200):
    k = rng.randint(0, 40); m = rng.randint(1, 20); n = rng.randint(k, k + m + 3)
    assert close(MySolutionAnswer().new21Game(n, k, m), float(f_values(n, k, m)[0]))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [39]:
def final_scores_answer(k, maxPts):
    if k == 0:
        return [1.0]
    top = k + maxPts - 1
    p = [0.0] * (top + 1)
    p[0] = 1.0
    window = 1.0
    for i in range(1, top + 1):
        p[i] = window / maxPts
        if i < k:
            window += p[i]
        if 0 <= i - maxPts < k:
            window -= p[i - maxPts]
    return p[k:]


fs = final_scores_answer(17, 10)
assert len(fs) == 10 and abs(sum(fs) - 1) < 1e-9 and abs(sum(fs[:5]) - 0.73278) < 1e-5
assert final_scores_answer(0, 5) == [1.0]
exact = final_distribution(17, 10)
assert all(abs(fs[i] - float(exact[17 + i])) < 1e-12 for i in range(10))
rng = random.Random(5)
for _ in range(100):
    k = rng.randint(1, 30); m = rng.randint(1, 15)
    fs = final_scores_answer(k, m)
    ex = final_distribution(k, m)
    assert len(fs) == m and all(abs(fs[i] - float(ex[k + i])) < 1e-12 for i in range(m))
# so với Monte Carlo: P(điểm cuối = k + m - 1) thường nhỏ nhất khi k lớn? chỉ kiểm tổng = 1
print("Bài 5 OK: phân phối điểm cuối khớp Fraction và tổng bằng 1")

Bài 5 OK: phân phối điểm cuối khớp Fraction và tổng bằng 1


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (xác suất với bước ngẫu nhiên đều `[1, m]`, tổng trượt):**

```python
class Solution:
    def new21Game(self, n: int, k: int, maxPts: int) -> float:
        if k == 0 or n >= k + maxPts - 1:
            return 1.0
        f = [0.0] * (k + maxPts)
        for x in range(k, n + 1):
            f[x] = 1.0
        s = float(sum(f[k:k + maxPts]))           # S(k-1)
        for x in range(k - 1, -1, -1):
            f[x] = s / maxPts
            s += f[x] - f[x + maxPts]             # thêm f[x], bớt f[x + maxPts]
        return f[0]
```

**Checklist trước khi submit:**

- [ ] Dừng khi **`≥ k`** (điểm `k` đã dừng): `f[x] = 1` cho `x ∈ [k, n]`?
- [ ] Chia cho `maxPts` (trung bình, không phải tổng)?
- [ ] Cửa sổ trượt: thêm `f[x]`, bớt `f[x + maxPts]` (đúng đầu xa)?
- [ ] Ca đặc biệt `k == 0` và `n >= k + maxPts - 1` trả `1.0`?
- [ ] Mảng đủ dài `k + maxPts` (không truy cập ngoài chỉ số)?
- [ ] Tự test số thực bằng dung sai (hoặc `Fraction`), không `==`?
- [ ] Đã thử `(10,1,10) → 1`, `(6,1,10) → 0.6`, `(21,17,10) → 0.73278`, `(n,k,1)`, `k = 0`?
- [ ] Đúng tên `class Solution`, `new21Game(self, n, k, maxPts)`, không in debug.